## Sentiment Analysis

In this exercise we use the IMDb-dataset, which we will use to perform a sentiment analysis. The code below assumes that the data is placed in the same folder as this notebook. We see that the reviews are loaded as a pandas dataframe, and print the beginning of the first few reviews.

In [8]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [3]:
reviews = pd.read_csv('reviews.txt', header=None)
labels = pd.read_csv('labels.txt', header=None)
Y = (labels=='positive').astype(np.int_)

print(type(reviews))
print(reviews.head())

<class 'pandas.core.frame.DataFrame'>
                                                   0
0  bromwell high is a cartoon comedy . it ran at ...
1  story of a man who has unnatural feelings for ...
2  homelessness  or houselessness as george carli...
3  airport    starts as a brand new luxury    pla...
4  brilliant over  acting by lesley ann warren . ...


**(a)** Split the reviews and labels in test, train and validation sets. The train and validation sets will be used to train your model and tune hyperparameters, the test set will be saved for testing. Use the `CountVectorizer` from `sklearn.feature_extraction.text` to create a Bag-of-Words representation of the reviews. Only use the 10,000 most frequent words (use the `max_features`-parameter of `CountVectorizer`).

In [4]:
# Flatten Y to a 1D array
Y = Y.values.ravel()

# Split the data: 60% train, 20% validation, 20% test
X_train_val, X_test, y_train_val, y_test = train_test_split(reviews[0], Y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.25, random_state=42)  # 0.25 * 0.8 = 0.2

# Initialize CountVectorizer with max 10,000 features
vectorizer = CountVectorizer(max_features=10000)

# Fit and transform the training data; transform the validation and test data
X_train_bow = vectorizer.fit_transform(X_train)
X_val_bow = vectorizer.transform(X_val)
X_test_bow = vectorizer.transform(X_test)

# Optional: Print shape of datasets
print(f"Train set shape: {X_train_bow.shape}")
print(f"Validation set shape: {X_val_bow.shape}")
print(f"Test set shape: {X_test_bow.shape}")

Train set shape: (15000, 10000)
Validation set shape: (5000, 10000)
Test set shape: (5000, 10000)


**(b)** Explore the representation of the reviews. How is a single word represented? How about a whole review?

In [5]:
# Each column in the Bag-of-Words matrix corresponds to a word.
# Get the first 10 words from the vectorizer's vocabulary
print("First 10 words in the vocabulary:")
print(vectorizer.get_feature_names_out()[:10])

# Let's inspect one review from the training set
review_example = X_train.iloc[0]
print("\nOriginal review text:")
print(review_example)

# Convert this review to its Bag-of-Words representation (sparse vector)
review_vector = vectorizer.transform([review_example])
print("\nBoW vector (non-zero entries):")
print(review_vector)

# Display the actual words present in the review according to BoW
# Each non-zero index in the vector corresponds to a word used in the review
word_indices = review_vector.indices
words_in_review = [vectorizer.get_feature_names_out()[i] for i in word_indices]

print("\nWords present in the review (according to BoW):")
print(words_in_review)

First 10 words in the vocabulary:
['abandon' 'abandoned' 'abby' 'abc' 'abducted' 'abilities' 'ability'
 'able' 'aboard' 'abominable']

Original review text:
  birth of the beatles   for being a us television movie  released in the fall of     has actually been  so far the best movie which tells the tale of the the four lads from liverpool that revolutionized the music industry and the world . as told by the point of view of former beatle pete best . the performance from the entire cast is excellent but  most especially the performance by stephen mackenna as john lennon and rod culbertson as paul mccartney . the film was produced by a legend of the rock and roll era  mr dick clark . who a year earlier in     had produced another tv movie  that has stood the test of time starring  kurt rusell  in the lead role about another musical legend  elvis  . that movie was directed by an unknown director named  john carpenter  who went on to direct other successful movies such as  halloween    esc

**(c)** Train a neural network with a single hidden layer on the dataset, tuning the relevant hyperparameters to optimize accuracy. 

In [7]:
# Define different hyperparameter combinations to try
hidden_layer_sizes = [(64,), (128,), (256,)]
alphas = [0.0001, 0.001, 0.01]
learning_rates = ['constant', 'adaptive']

best_model = None
best_val_acc = 0
results = []

# Try all combinations of hyperparameters
for size in hidden_layer_sizes:
    for alpha in alphas:
        for lr in learning_rates:
            print(f"Training model: hidden_layer={size}, alpha={alpha}, learning_rate={lr}")
            clf = MLPClassifier(hidden_layer_sizes=size,
                                alpha=alpha,
                                learning_rate=lr,
                                max_iter=50,  # Increase this for more training
                                random_state=42,
                                verbose=False)
            clf.fit(X_train_bow, y_train)
            val_preds = clf.predict(X_val_bow)
            val_acc = accuracy_score(y_val, val_preds)
            results.append((size, alpha, lr, val_acc))
            print(f"Validation Accuracy: {val_acc:.4f}")

            # Keep track of best model
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                best_model = clf

# Show the best result
print("\nBest Model Parameters:")
print(best_model)

# Evaluate best model on test set
test_preds = best_model.predict(X_test_bow)
test_acc = accuracy_score(y_test, test_preds)
print(f"\nTest Accuracy: {test_acc:.4f}")


Training model: hidden_layer=(64,), alpha=0.0001, learning_rate=constant


C:\Users\doomz\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (10) reached and the optimization hasn't converged yet.
  warnings.warn(


Validation Accuracy: 0.8792
Training model: hidden_layer=(64,), alpha=0.0001, learning_rate=adaptive


C:\Users\doomz\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (10) reached and the optimization hasn't converged yet.
  warnings.warn(


Validation Accuracy: 0.8792
Training model: hidden_layer=(64,), alpha=0.001, learning_rate=constant


...

Validation Accuracy: 0.8852
Training model: hidden_layer=(256,), alpha=0.01, learning_rate=constant


C:\Users\doomz\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (10) reached and the optimization hasn't converged yet.
  warnings.warn(


Validation Accuracy: 0.8816
Training model: hidden_layer=(256,), alpha=0.01, learning_rate=adaptive


C:\Users\doomz\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (10) reached and the optimization hasn't converged yet.
  warnings.warn(


Validation Accuracy: 0.8816

Best Model Parameters:
MLPClassifier(hidden_layer_sizes=(256,), max_iter=10, random_state=42)

Test Accuracy: 0.8720


**(d)** Test your sentiment-classifier on the test set.

In [9]:
# Predict the labels for the test set
y_test_pred = best_model.predict(X_test_bow)

# Compute accuracy
test_accuracy = accuracy_score(y_test, y_test_pred)
print(f"Test Accuracy: {test_accuracy:.4f}")

# Detailed performance report
print("\nClassification Report:")
print(classification_report(y_test, y_test_pred, target_names=['Negative', 'Positive']))

# Confusion matrix
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))


Test Accuracy: 0.8720

Classification Report:
              precision    recall  f1-score   support

    Negative       0.87      0.87      0.87      2492
    Positive       0.87      0.87      0.87      2508

    accuracy                           0.87      5000
   macro avg       0.87      0.87      0.87      5000
weighted avg       0.87      0.87      0.87      5000

Confusion Matrix:
[[2174  318]
 [ 322 2186]]


**(e)** Use the classifier to classify a few sentences you write yourselves. 

In [10]:
# Example custom sentences
custom_sentences = [
    "I absolutely loved this movie! It was fantastic.",
    "What a waste of time. The plot was boring and predictable.",
    "It was okay, not great but not terrible either.",
    "The acting was brilliant, but the story made no sense.",
    "Terrible. I walked out halfway through."
]

# Transform custom sentences to Bag-of-Words format using the trained vectorizer
custom_bow = vectorizer.transform(custom_sentences)

# Predict sentiments
predictions = best_model.predict(custom_bow)

# Convert binary predictions back to labels
labels = ['Negative', 'Positive']
for sentence, pred in zip(custom_sentences, predictions):
    print(f"Sentence: {sentence}\nPredicted Sentiment: {labels[pred]}\n")


Sentence: I absolutely loved this movie! It was fantastic.
Predicted Sentiment: Positive

Sentence: What a waste of time. The plot was boring and predictable.
Predicted Sentiment: Negative

Sentence: It was okay, not great but not terrible either.
Predicted Sentiment: Negative

Sentence: The acting was brilliant, but the story made no sense.
Predicted Sentiment: Positive

Sentence: Terrible. I walked out halfway through.
Predicted Sentiment: Negative

